# Session 23: Distributions and Outliers

Data Science with Python, Level 2, Stage 3: Exploring Data

## Learning Objectives

By the end of this session you will be able to:

1. Describe a distribution: centre, spread, shape.
2. Explain why mean and median can differ.
3. Use quantiles and the IQR rule.
4. Find outliers with z-scores.
5. Decide whether to keep, fix, cap or remove.
6. Use a log scale for skewed data.

## Where We Are

In Session 22 the box plots showed dots beyond the whiskers. Those are **outliers**: values far away from the rest. Some are mistakes, such as an age of 230. Others are real but rare, such as a luxury flat. Both can change an average, distort a chart and mislead a model.

Today we use a practice dataset of **800 apartment sales** in a fictional city, with prices in thousands. We describe the shape of the prices, find the unusual ones with clear rules, and decide what to do with them.

## Setup

Run this cell first. The apartment data has one row per sale, with the district, size, rooms, floor, building age, distance to the centre and price in thousands.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
flats = pd.read_csv(DATA_URL + "apartments.csv")

## 1. Describing a Distribution

In [ ]:
sns.histplot(data=flats, x="price_k", bins=40)
plt.title("Apartment prices (thousands)")
plt.show()

In [ ]:
price = flats["price_k"]
print("Mean:  ", round(price.mean(), 1))
print("Median:", price.median())
print("Std:   ", round(price.std(), 1))
print("Skew:  ", round(price.skew(), 2))

## 2. Quantiles and the IQR Rule

A **quantile** splits the sorted values: the 0.25 quantile, called the first quartile Q1, has a quarter of the values below it. The median is the 0.5 quantile. The **interquartile range** (IQR) is Q3 minus Q1, the width of the middle half of the data.

The **IQR rule**, used by every box plot, calls a value an outlier when it lies more than 1.5 IQR below Q1 or above Q3.

In [ ]:
q1, q3 = price.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr

print(f"Q1 {q1:.1f}, Q3 {q3:.1f}, IQR {iqr:.1f}")
print(f"Fences: {low:.1f} to {high:.1f}")
outliers = flats[(price < low) | (price > high)]
print(len(outliers), "outliers")

In [ ]:
outliers.sort_values("price_k", ascending=False)[
    ["district", "area_m2", "rooms", "price_k"]].head(8)

## 3. z-Scores

A **z-score** measures how far a value is from the mean, in standard deviations: `z = (x - mean) / std`. A z-score of 2 means two standard deviations above the mean. A common rule calls a value an outlier when its z-score is above 3 or below -3. This works best for distributions that are roughly symmetric.

In [ ]:
flats["price_z"] = (price - price.mean()) / price.std()

extreme = flats[flats["price_z"].abs() > 3]
print(len(extreme), "flats with |z| > 3")
extreme[["district", "area_m2", "price_k", "price_z"]].round(2)

## 4. Context Matters: Outliers per Group

In [ ]:
flats["price_per_m2"] = flats["price_k"] * 1000 / flats["area_m2"]

order = flats.groupby("district")["price_per_m2"].median().sort_values().index
sns.boxplot(data=flats, x="district", y="price_per_m2", order=order)
plt.title("Price per square metre by district")
plt.show()

In [ ]:
grp = flats.groupby("district")["price_per_m2"]
q1g = grp.transform(lambda s: s.quantile(0.25))
q3g = grp.transform(lambda s: s.quantile(0.75))
flats["unusual"] = flats["price_per_m2"] > q3g + 1.5 * (q3g - q1g)

print(flats["unusual"].sum(), "unusual flats for their district")
flats.loc[flats["unusual"], ["district", "area_m2", "price_k"]].head(6)

## 5. What to Do with Outliers

| Situation | Action | Code |
| --- | --- | --- |
| Impossible value (age 230) | Fix it, or set to missing | `df.loc[mask, col] = np.nan` |
| Real but extreme, and it matters | Keep it; use the median and robust charts | `df[col].median()` |
| Real, but it distorts a model | Cap it at a limit (winsorise) | `df[col].clip(upper=high)` |
| Outside the question you study | Remove it, and say so | `df[~mask]` |

Never delete outliers just to make the data look nicer. Write down every decision and its reason.

In [ ]:
capped = price.clip(upper=high)
print("Mean before:", round(price.mean(), 1), "after capping:", round(capped.mean(), 1))
print("Median before:", price.median(), "after capping:", capped.median())

## 6. Log Scales for Skewed Data

In [ ]:
flats["log_price"] = np.log(flats["price_k"])
print("Skew of price:", round(flats["price_k"].skew(), 2))
print("Skew of log price:", round(flats["log_price"].skew(), 2))

sns.histplot(data=flats, x="log_price", bins=40)
plt.title("Log of apartment prices")
plt.show()

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Biased summary | Reporting the mean of strongly skewed data | Report the median, or both. |
| Data destroyed | Deleting every value the IQR rule flags | Look at each outlier and decide with reasons. |
| Unfair test | Judging a value against all rows | Compare with similar rows, such as the same district. |
| Wrong rule | Using z-scores on very skewed data | Use the IQR rule, or z-scores of the log. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Area Distribution

Draw a histogram of `area_m2` and print its mean, median and skew. Is it skewed like the prices?

In [ ]:
# Write your code below

### In-Class Exercise 2: A Reusable Rule

Write `iqr_outliers(series)` that returns a mask of the values outside the IQR fences. Use it to count the outliers in `area_m2` and in `distance_center_km`.

In [ ]:
# Write your code below

### In-Class Exercise 3: Weekly Hours

Load `employees.csv`. Find employees whose salary has a z-score above 2.5 within their own city. Hint: compute the mean and std per city with `transform`.

In [ ]:
# Write your code below

### In-Class Exercise 4: Cap and Compare

Cap the rentals in `bike_rentals.csv` at their 1st and 99th percentiles with `clip(lower=..., upper=...)`, and compare the mean and standard deviation before and after.

In [ ]:
# Write your code below

## Summary

- Describe a distribution by its centre, spread and shape.
- Skewed data: the mean follows the tail; the median does not.
- The IQR rule: outside Q1 - 1.5 IQR or Q3 + 1.5 IQR.
- z-scores: more than 3 standard deviations from the mean.
- Judge outliers within their group, then fix, keep, cap or remove.
- A log transform tames right-skewed data.

## What's Next

Next session you will run a complete exploratory analysis on a famous real dataset: the passengers of the Titanic. You will use every tool from Stages 2 and 3 to answer one question: who survived, and why?